# Simulating Constitutive Processes of semantic change within heterogeneous populations of speakers

In [3]:
# basic imports
import os
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import statsmodels.formula.api as smf

# project code imports
from mod.one_hot_agent import *
from mod.plot import *
from mod.network import *

##### Hyper parameters

In [4]:
model_version = 'sn-1hot'

In [5]:
turns = 300
no_agents = 25
no_connections = 5
add_vocab_in = .98
semantic_features = 3
starting_observations = 5
starting_uncertainty = .2
words_per_semantic_feature = 100
new_environment_prob = .25
enforce_word_feature_mapping = False
no_simulations = 100

In [6]:
model_path = os.path.join('html',model_version)
if not os.path.exists(model_path):
    os.mkdir(model_path)

Simplifying the way episodes are run :)

In [7]:
def episode(
        net,
        starting_env,
        new_environment_prob: None|float=None,
        new_vocab_round_prob: None|float=None,
        stochastic_environment_updates: bool=False,
        lexicon_smoothening: float=1.
):

    if new_environment_prob:
        new_env_prob = torch.rand(size=(1,))
        if new_env_prob > new_environment_prob:
            starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

    new_vocab_round = False
    if new_vocab_round_prob:
        new_vocab_prob = torch.rand(size=(1,))
        if new_vocab_prob > new_vocab_round_prob:
            new_vocab_round = True


    env = starting_env.sample()
    if stochastic_environment_updates:
        starting_env.loc = env

    round_feature = torch.zeros(size=env.shape)
    round_feature[:,np.random.choice(env.shape[-1])] = 1.
    env = round_feature * env

    net.interaction(env, lexicon_smoothening, new_vocab_round)

    return starting_env, net

## Randomly generated environment

In [6]:
env_name = 'Randomly-Generated-Environment'

In [7]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [8]:
vocab_dif, H_dif = [], []

In [9]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(net, starting_env)

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

#### Vocab difference stats

In [10]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.047
Model:                            OLS   Adj. R-squared:                  0.047
Method:                 Least Squares   F-statistic:                 3.725e+04
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:27:10   Log-Likelihood:            -3.4792e+06
No. Observations:              750000   AIC:                         6.958e+06
Df Residuals:                  749998   BIC:                         6.958e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        572.7359      0.058   9884.

In [11]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Randomly-Generated-Environment,572.735947,0.057945,9884.060915,0.0
interaction_no,Randomly-Generated-Environment,-0.064412,0.000334,-193.015462,0.0


In [13]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [14]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.242
Model:                            OLS   Adj. R-squared:                  0.242
Method:                 Least Squares   F-statistic:                 2.396e+05
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:28:08   Log-Likelihood:            -8.6334e+05
No. Observations:              750000   AIC:                         1.727e+06
Df Residuals:                  749998   BIC:                         1.727e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        872.7743      0.002   4.93e

In [15]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Randomly-Generated-Environment,872.774311,0.001771,492748.886025,0.0
interaction_no,Randomly-Generated-Environment,0.004993,0.000010,489.481317,0.0


In [16]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [17]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [18]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,577.307484
1,1,2,577.202490
2,1,3,577.072317
3,1,4,576.933981
4,1,5,576.797937


In [19]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [20]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [21]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [22]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)

## Random environment and introduction of new terms

In [23]:
env_name = 'Randomly-Generated-Environment-and-Introducing-Novel-Terms'

In [24]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [25]:
vocab_dif, H_dif = [], []

In [26]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(
            net,
            starting_env,
            new_vocab_round_prob=add_vocab_in
        )

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

#### Vocab difference stats

In [27]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.033
Model:                            OLS   Adj. R-squared:                  0.033
Method:                 Least Squares   F-statistic:                 2.538e+04
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:29:53   Log-Likelihood:            -3.4831e+06
No. Observations:              750000   AIC:                         6.966e+06
Df Residuals:                  749998   BIC:                         6.966e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        572.3743      0.058   9827.

In [28]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Randomly-Generated-Environment-and-Introducing...,572.374300,0.058243,9827.293544,0.0
interaction_no,Randomly-Generated-Environment-and-Introducing...,-0.053436,0.000335,-159.305266,0.0


In [29]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [30]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.601
Model:                            OLS   Adj. R-squared:                  0.601
Method:                 Least Squares   F-statistic:                 1.129e+06
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:30:19   Log-Likelihood:            -1.6615e+06
No. Observations:              750000   AIC:                         3.323e+06
Df Residuals:                  749998   BIC:                         3.323e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        873.4672      0.005    1.7e

In [31]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Randomly-Generated-Environment-and-Introducing...,873.467221,0.005134,170129.559071,0.0
interaction_no,Randomly-Generated-Environment-and-Introducing...,0.031421,0.000030,1062.674167,0.0


In [32]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [33]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [34]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,576.402412
1,1,2,576.267106
2,1,3,576.166225
3,1,4,576.021958
4,1,5,575.930424


In [35]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [36]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [37]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [38]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)

## Changing/Upheaval environment

In [39]:
env_name = 'Suddenly-Changing-Environment'

In [40]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [41]:
vocab_dif, H_dif = [], []

In [42]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(
            net,
            starting_env,
            new_environment_prob=new_environment_prob
        )

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

#### Vocab difference stats

In [43]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.049
Model:                            OLS   Adj. R-squared:                  0.049
Method:                 Least Squares   F-statistic:                 3.847e+04
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:32:11   Log-Likelihood:            -3.4495e+06
No. Observations:              750000   AIC:                         6.899e+06
Df Residuals:                  749998   BIC:                         6.899e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        571.2728      0.056   1.03e

In [44]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Suddenly-Changing-Environment,571.272841,0.055696,10257.034630,0.0
interaction_no,Suddenly-Changing-Environment,-0.062913,0.000321,-196.140064,0.0


In [45]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [46]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.428
Model:                            OLS   Adj. R-squared:                  0.428
Method:                 Least Squares   F-statistic:                 5.613e+05
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:32:20   Log-Likelihood:            -1.3820e+06
No. Observations:              750000   AIC:                         2.764e+06
Df Residuals:                  749998   BIC:                         2.764e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        873.1058      0.004   2.47e

In [47]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Suddenly-Changing-Environment,873.105839,0.003537,246849.004170,0.0
interaction_no,Suddenly-Changing-Environment,0.015261,0.000020,749.173962,0.0


In [48]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [49]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [50]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,571.833390
1,1,2,571.643576
2,1,3,571.514313
3,1,4,571.355098
4,1,5,571.223185


In [51]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [52]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [53]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [54]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)

## Changing/Upheaval environment and introduction of new terms

In [55]:
env_name = 'Suddenly-Changing-Environment-and-Introducing-Novel-Terms'

In [56]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [57]:
vocab_dif, H_dif = [], []

In [58]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(
            net,
            starting_env,
            new_environment_prob=new_environment_prob,
            new_vocab_round_prob=add_vocab_in
        )

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

#### Vocab difference stats

In [59]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.043
Model:                            OLS   Adj. R-squared:                  0.043
Method:                 Least Squares   F-statistic:                 3.386e+04
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:34:22   Log-Likelihood:            -3.4478e+06
No. Observations:              750000   AIC:                         6.896e+06
Df Residuals:                  749998   BIC:                         6.896e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        573.1143      0.056   1.03e

In [60]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Suddenly-Changing-Environment-and-Introducing-...,573.114327,0.05557,10313.459069,0.0
interaction_no,Suddenly-Changing-Environment-and-Introducing-...,-0.058889,0.00032,-184.008937,0.0


In [61]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [62]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.628
Model:                            OLS   Adj. R-squared:                  0.628
Method:                 Least Squares   F-statistic:                 1.268e+06
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:34:31   Log-Likelihood:            -1.6964e+06
No. Observations:              750000   AIC:                         3.393e+06
Df Residuals:                  749998   BIC:                         3.393e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        873.1800      0.005   1.62e

In [63]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Suddenly-Changing-Environment-and-Introducing-...,873.180014,0.005379,162343.600173,0.0
interaction_no,Suddenly-Changing-Environment-and-Introducing-...,0.034887,0.000031,1126.263070,0.0


In [64]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [65]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [66]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,582.159777
1,1,2,582.059291
2,1,3,581.951019
3,1,4,581.817293
4,1,5,581.684860


In [67]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [68]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [69]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [70]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)

## Stochastic environment

In [8]:
env_name = 'Stochastically-Changing-Environment'

In [9]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [10]:
vocab_dif, H_dif = [], []

In [11]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(
            net,
            starting_env,
            stochastic_environment_updates=True
        )

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i, val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

In [20]:
H_dif = H_dif.loc[~np.isinf(H_dif['delta'])]

#### Vocab difference stats

In [21]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.190
Model:                            OLS   Adj. R-squared:                  0.190
Method:                 Least Squares   F-statistic:                 1.757e+05
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:52:25   Log-Likelihood:            -6.6842e+06
No. Observations:              750000   AIC:                         1.337e+07
Df Residuals:                  749998   BIC:                         1.337e+07
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept       1082.0093      4.158    260.

In [22]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Stochastically-Changing-Environment,1082.009323,4.157879,260.231047,0.0
interaction_no,Stochastically-Changing-Environment,10.037440,0.023946,419.175550,0.0


In [23]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [24]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.179
Model:                            OLS   Adj. R-squared:                  0.179
Method:                 Least Squares   F-statistic:                 3.337e+04
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:52:51   Log-Likelihood:            -9.3180e+05
No. Observations:              153567   AIC:                         1.864e+06
Df Residuals:                  153565   BIC:                         1.864e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        908.8331      0.458   1986.

In [25]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Stochastically-Changing-Environment,908.833117,0.457566,1986.233917,0.0
interaction_no,Stochastically-Changing-Environment,0.528256,0.002892,182.677505,0.0


In [26]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [27]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [28]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,592.346411
1,1,2,612.513807
2,1,3,629.376433
3,1,4,642.480811
4,1,5,663.579561


In [29]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [84]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [87]:
# H_dif.head()

In [30]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [31]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)

## Stochastic environment and introduction of new terms

In [32]:
env_name = 'Stochastically-Changing-Environment-and-Introducing-Novel-Terms'

In [33]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [34]:
vocab_dif, H_dif = [], []

In [35]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(
            net,
            starting_env,
            stochastic_environment_updates=True,
            new_vocab_round_prob=add_vocab_in
        )

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

In [36]:
np.isinf(H_dif).sum()

simulation             0
agent                  0
interaction_no         0
delta             695498
dtype: int64

In [37]:
H_dif = H_dif.loc[~np.isinf(H_dif['delta'])]

#### Vocab difference stats

In [38]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.180
Model:                            OLS   Adj. R-squared:                  0.180
Method:                 Least Squares   F-statistic:                 1.651e+05
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:55:27   Log-Likelihood:            -6.9423e+06
No. Observations:              750000   AIC:                         1.388e+07
Df Residuals:                  749998   BIC:                         1.388e+07
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept       1162.8272      5.866    198.

In [39]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Stochastically-Changing-Environment-and-Introd...,1162.827197,5.866298,198.221638,0.0
interaction_no,Stochastically-Changing-Environment-and-Introd...,13.726851,0.033785,406.304519,0.0


In [40]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [41]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.245
Model:                            OLS   Adj. R-squared:                  0.245
Method:                 Least Squares   F-statistic:                 1.772e+04
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:55:37   Log-Likelihood:            -3.3032e+05
No. Observations:               54502   AIC:                         6.606e+05
Df Residuals:                   54500   BIC:                         6.607e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        908.0494      0.601   1510.

In [42]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Stochastically-Changing-Environment-and-Introd...,908.049413,0.600989,1510.925750,0.0
interaction_no,Stochastically-Changing-Environment-and-Introd...,0.709488,0.005330,133.102574,0.0


In [43]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [44]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [45]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,592.756907
1,1,2,615.704919
2,1,3,639.931479
3,1,4,671.653960
4,1,5,706.506039


In [46]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [47]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [48]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [49]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)

## Upheaval + stochastic environment

In [105]:
env_name = 'Suddenly-and-Stochastically-Changing-Environment'

In [106]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [107]:
vocab_dif, H_dif = [], []

In [108]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(
            net,
            starting_env,
            new_environment_prob=new_environment_prob,
            stochastic_environment_updates=True
        )

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

#### Vocab difference stats

In [109]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.047
Model:                            OLS   Adj. R-squared:                  0.047
Method:                 Least Squares   F-statistic:                 3.734e+04
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:40:50   Log-Likelihood:            -3.4517e+06
No. Observations:              750000   AIC:                         6.903e+06
Df Residuals:                  749998   BIC:                         6.903e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        572.4161      0.056   1.02e

In [110]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Suddenly-and-Stochastically-Changing-Environment,572.416101,0.055859,10247.517696,0.0
interaction_no,Suddenly-and-Stochastically-Changing-Environment,-0.062163,0.000322,-193.233771,0.0


In [111]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [112]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.421
Model:                            OLS   Adj. R-squared:                  0.421
Method:                 Least Squares   F-statistic:                 5.456e+05
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:41:02   Log-Likelihood:            -1.4038e+06
No. Observations:              750000   AIC:                         2.808e+06
Df Residuals:                  749998   BIC:                         2.808e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        873.1933      0.004    2.4e

In [113]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Suddenly-and-Stochastically-Changing-Environment,873.193348,0.003641,239813.245921,0.0
interaction_no,Suddenly-and-Stochastically-Changing-Environment,0.015490,0.000021,738.669752,0.0


In [114]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [115]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [116]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,572.374309
1,1,2,572.249282
2,1,3,572.120159
3,1,4,571.996926
4,1,5,571.873177


In [117]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [118]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [119]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [120]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)

## Upheaval + stochastic environment, plus introduction of new terms

In [121]:
env_name = 'Suddenly-and-Stochastically-Changing-Environment-and-Introducing-Novel-Terms'

In [122]:
starting_env = torch.distributions.MultivariateNormal(torch.randn(size=(1,semantic_features)), covariance_matrix=torch.eye(semantic_features) * .2)

In [123]:
vocab_dif, H_dif = [], []

In [124]:
for sim in tqdm(range(no_simulations)):

    net = social_network(
        k_agents=no_agents,
        vocab_size=words_per_semantic_feature,
        connections_per_agent=no_connections,
        semantic_dimensions=semantic_features,
        starting_observations=starting_observations,
        starting_uncertainty=starting_uncertainty,
        enforcing=enforce_word_feature_mapping
    )

    for interaction in range(turns):

        starting_env, net = episode(
            net,
            starting_env,
            new_environment_prob=new_environment_prob,
            new_vocab_round_prob=add_vocab_in,
            stochastic_environment_updates=True
        )

        vocab_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.similarity_heatmap().mean(dim=-1).tolist())]
        H_dif += [{'simulation': sim, 'agent':i + 1, 'interaction_no': interaction + 1, 'delta': val} for i,val in enumerate(net.network_entropy().tolist())]

vocab_dif = pd.DataFrame(vocab_dif)
H_dif = pd.DataFrame(H_dif)

  0%|          | 0/100 [00:00<?, ?it/s]

#### Vocab difference stats

In [125]:
mod = smf.ols('delta ~ interaction_no', data=vocab_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.043
Model:                            OLS   Adj. R-squared:                  0.043
Method:                 Least Squares   F-statistic:                 3.385e+04
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:42:46   Log-Likelihood:            -3.4607e+06
No. Observations:              750000   AIC:                         6.921e+06
Df Residuals:                  749998   BIC:                         6.921e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        573.4637      0.057   1.01e

In [126]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Suddenly-and-Stochastically-Changing-Environme...,573.463683,0.056530,10144.436288,0.0
interaction_no,Suddenly-and-Stochastically-Changing-Environme...,-0.059896,0.000326,-183.978073,0.0


In [127]:
if os.path.exists(os.path.join(model_path, 'vocab-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'vocab-dif.csv'),
        encoding='utf-8',
    )

#### Entropy change stats

In [128]:
mod = smf.ols('delta ~ interaction_no', data=H_dif)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.660
Model:                            OLS   Adj. R-squared:                  0.660
Method:                 Least Squares   F-statistic:                 1.458e+06
Date:                Tue, 10 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:42:59   Log-Likelihood:            -1.6586e+06
No. Observations:              750000   AIC:                         3.317e+06
Df Residuals:                  749998   BIC:                         3.317e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        872.9776      0.005   1.71e

In [129]:
odf = pd.DataFrame()
odf['coefs'] = res.params
odf['se'] = res.bse
odf['stat'] = res.tvalues
odf['p'] = res.pvalues
odf['env_type'] = env_name

odf = odf[['env_type'] + [col for col in list(odf) if col not in ['env_type']]]
odf.head()

,env_type,coefs,se,stat,p
Intercept,Suddenly-and-Stochastically-Changing-Environme...,872.977631,0.005114,170701.500382,0.0
interaction_no,Suddenly-and-Stochastically-Changing-Environme...,0.035562,0.000029,1207.434711,0.0


In [130]:
if os.path.exists(os.path.join(model_path, 'H-dif.csv')):
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        header=False,
        encoding='utf-8',
        mode='a'
    )
else:
    odf.to_csv(
        os.path.join(model_path, 'H-dif.csv'),
        encoding='utf-8',
    )

#### Visualizations

In [131]:
vocab_dif = vocab_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()
H_dif = H_dif[['agent', 'interaction_no', 'delta']].groupby(by=['agent', 'interaction_no']).agg('mean').reset_index()

In [132]:
vocab_dif.head()

,agent,interaction_no,delta
0,1,1,576.426937
1,1,2,576.346358
2,1,3,576.251940
3,1,4,576.089304
4,1,5,575.994616


In [133]:
fig = multi_agent_update_plot_from_df(
    vocab_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='Δ P(w|m)',
    xaxis_title='interactions'
)
fig.show()

In [134]:
fig.write_html(
    os.path.join(model_path,env_name+'.html')
)

In [135]:
fig = multi_agent_update_plot_from_df(
    H_dif,
    time_col='interaction_no',
    dif_col='delta'
)

fig.update_layout(
    # title='Dyadic interaction in a random environment',
    yaxis_title='H(vocab)',
    xaxis_title='interactions'
)
fig.show()

In [136]:
fig.write_html(
    os.path.join(model_path,env_name+'-H.html')
)